## Cell 1

In [1]:
import os
import json

import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings('ignore')


## Cell 2

In [4]:
DATA_PATH_PARQUET = "returns_dataset.parquet"
DATA_PATH_CSV = "returns_dataset.csv"

MODEL_OUT = "return_risk_scorer.pkl"
ARTIFACTS_OUT = "return_risk_preprocessing_artifacts.pkl"
CALIB_PREDICTIONS_OUT = "return_risk_calib_predictions.csv"
TEST_PREDICTIONS_OUT = "return_risk_test_predictions.csv"
RESULTS_OUT = "return_risk_results.json"

FEATURES = [
    'Time_to_Return_Days',
    'Item_Category',
    'Item_Margin_USD',
    'Claim_Type',
    'Customer_LTV',
    'Returns_Count_Last_90D',
    'Prior_Confirmed_Fraud_Count'
]
CATEGORICAL_FEATURES = ['Item_Category', 'Claim_Type']
TARGET = 'Is_Fraud'

TRAIN_FRACTION = 0.70
CALIB_FRACTION = 0.15

RANDOM_STATE = 42


## Cell 3

In [5]:
if os.path.exists(DATA_PATH_PARQUET):
    df = pd.read_parquet(DATA_PATH_PARQUET)
elif os.path.exists(DATA_PATH_CSV):
    df = pd.read_csv(DATA_PATH_CSV)
else:
    raise FileNotFoundError(f"Neither {DATA_PATH_PARQUET} nor {DATA_PATH_CSV} found.")

print(f"Loaded {len(df):,} rows | Fraud rate: {df[TARGET].mean() * 100:.2f}%")


Loaded 15,100 rows | Fraud rate: 11.11%


## Cell 4

In [6]:
df = df.sort_values('ReturnDT').reset_index(drop=True)

n = len(df)
train_end = int(n * TRAIN_FRACTION)
calib_end = int(n * (TRAIN_FRACTION + CALIB_FRACTION))

train_df = df.iloc[:train_end].copy()
calib_df = df.iloc[train_end:calib_end].copy()
test_df = df.iloc[calib_end:].copy()

print(f"Temporal Split -> Train: {len(train_df):,} | Calib: {len(calib_df):,} | Test: {len(test_df):,}")
print(f"Train ReturnDT max: {train_df['ReturnDT'].max()}")
print(f"Calib ReturnDT min/max: {calib_df['ReturnDT'].min()} / {calib_df['ReturnDT'].max()}")
print(f"Test ReturnDT min: {test_df['ReturnDT'].min()}")


Temporal Split -> Train: 10,570 | Calib: 2,265 | Test: 2,265
Train ReturnDT max: 16373881
Calib ReturnDT min/max: 16377804 / 19803760
Test ReturnDT min: 19805944


## Cell 5

In [7]:
category_maps = {}

for col in CATEGORICAL_FEATURES:
    categories = pd.Index(sorted(train_df[col].dropna().unique()))
    category_maps[col] = categories.tolist()

    train_df[col] = pd.Categorical(train_df[col], categories=categories)
    calib_df[col] = pd.Categorical(calib_df[col], categories=categories)
    test_df[col] = pd.Categorical(test_df[col], categories=categories)

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_calib, y_calib = calib_df[FEATURES], calib_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]


## Cell 6

In [8]:
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
pos_weight = neg_count / pos_count

print(f"Train fraud rate: {y_train.mean() * 100:.2f}% | scale_pos_weight: {pos_weight:.2f}")


Train fraud rate: 11.38% | scale_pos_weight: 7.79


## Cell 7

In [9]:
model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    num_leaves=31,
    scale_pos_weight=pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_calib, y_calib)],
    eval_metric='average_precision',
    callbacks=[
        lgb.early_stopping(stopping_rounds=30, first_metric_only=True, verbose=False),
        lgb.log_evaluation(0)
    ]
)

print(f"Best iteration: {model.best_iteration_}")


[LightGBM] [Info] Number of positive: 1203, number of negative: 9367
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000428 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 565
[LightGBM] [Info] Number of data points in the train set: 10570, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.113813 -> initscore=-2.052374
[LightGBM] [Info] Start training from score -2.052374
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

## Cell 8

In [10]:
calib_probs = model.predict_proba(X_calib, num_iteration=model.best_iteration_)[:, 1]

precision_curve, recall_curve, thresholds = precision_recall_curve(y_calib, calib_probs)
f1_curve = np.where(
    (precision_curve + recall_curve) > 0,
    2 * precision_curve * recall_curve / (precision_curve + recall_curve),
    0.0
)

best_idx = np.argmax(f1_curve[:-1])
best_threshold = float(thresholds[best_idx])

print(f"Calibration PR-AUC: {average_precision_score(y_calib, calib_probs):.4f}")
print(f"Best-F1 threshold (from calibration only): {best_threshold:.4f}")
print(f"Calibration precision/recall/F1 at this threshold: "
      f"{precision_curve[best_idx]:.4f} / {recall_curve[best_idx]:.4f} / {f1_curve[best_idx]:.4f}")


Calibration PR-AUC: 0.8394
Best-F1 threshold (from calibration only): 0.8983
Calibration precision/recall/F1 at this threshold: 0.8157 / 0.7500 / 0.7815


## Cell 9

In [11]:
test_probs = model.predict_proba(X_test, num_iteration=model.best_iteration_)[:, 1]
test_preds = (test_probs >= best_threshold).astype(int)

roc_auc = roc_auc_score(y_test, test_probs)
pr_auc = average_precision_score(y_test, test_probs)
precision = precision_score(y_test, test_preds, zero_division=0)
recall = recall_score(y_test, test_preds, zero_division=0)
f1 = f1_score(y_test, test_preds, zero_division=0)
tn, fp, fn, tp = confusion_matrix(y_test, test_preds).ravel()

print("--- Held-Out Test Set (never touched before this cell) ---")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}")
print(f"Precision / Recall / F1 @ threshold={best_threshold:.4f}: "
      f"{precision:.4f} / {recall:.4f} / {f1:.4f}")
print(f"TN: {tn} | FP: {fp} | FN: {fn} | TP: {tp}")
print()
print(classification_report(y_test, test_preds, target_names=['Legitimate', 'Fraud']))


--- Held-Out Test Set (never touched before this cell) ---
ROC-AUC: 0.9541
PR-AUC:  0.8397
Precision / Recall / F1 @ threshold=0.8983: 0.8000 / 0.7029 / 0.7483
TN: 1984 | FP: 42 | FN: 71 | TP: 168

              precision    recall  f1-score   support

  Legitimate       0.97      0.98      0.97      2026
       Fraud       0.80      0.70      0.75       239

    accuracy                           0.95      2265
   macro avg       0.88      0.84      0.86      2265
weighted avg       0.95      0.95      0.95      2265



## Cell 10

In [12]:
feat_imp = pd.DataFrame({
    'Feature': FEATURES,
    'Gain_Importance': model.booster_.feature_importance(importance_type='gain'),
    'Split_Importance': model.booster_.feature_importance(importance_type='split')
}).sort_values('Gain_Importance', ascending=False)

print(feat_imp.to_string(index=False))


                    Feature  Gain_Importance  Split_Importance
            Item_Margin_USD     60815.217203              1202
Prior_Confirmed_Fraud_Count     40732.313780               195
        Time_to_Return_Days     24629.282065               975
               Customer_LTV     17342.873872               988
                 Claim_Type      6105.887809               148
              Item_Category      1759.429532                76
     Returns_Count_Last_90D       908.894376                69


## Cell 11

In [13]:
joblib.dump(model, MODEL_OUT)

artifacts = {
    'feature_columns': FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'category_maps': category_maps,
    'decision_threshold': best_threshold,
    'best_iteration': int(model.best_iteration_)
}
joblib.dump(artifacts, ARTIFACTS_OUT)

pd.DataFrame({
    'actual_fraud': y_calib.to_numpy(),
    'fraud_probability': calib_probs
}).to_csv(CALIB_PREDICTIONS_OUT, index=False)

pd.DataFrame({
    'actual_fraud': y_test.to_numpy(),
    'fraud_probability': test_probs
}).to_csv(TEST_PREDICTIONS_OUT, index=False)

results = {
    'n_train': int(len(train_df)),
    'n_calib': int(len(calib_df)),
    'n_test': int(len(test_df)),
    'scale_pos_weight': float(pos_weight),
    'best_iteration': int(model.best_iteration_),
    'decision_threshold': best_threshold,
    'calibration_metrics': {
        'pr_auc': float(average_precision_score(y_calib, calib_probs)),
        'roc_auc': float(roc_auc_score(y_calib, calib_probs))
    },
    'test_metrics': {
        'pr_auc': float(pr_auc),
        'roc_auc': float(roc_auc),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'true_positives': int(tp),
        'true_negatives': int(tn),
        'false_positives': int(fp),
        'false_negatives': int(fn)
    },
    'feature_importance_gain': feat_imp.set_index('Feature')['Gain_Importance'].to_dict()
}

with open(RESULTS_OUT, 'w') as f:
    json.dump(results, f, indent=4)

print(f"Model saved to: {MODEL_OUT}")
print(f"Artifacts saved to: {ARTIFACTS_OUT}")
print(f"Results saved to: {RESULTS_OUT}")


Model saved to: return_risk_scorer.pkl
Artifacts saved to: return_risk_preprocessing_artifacts.pkl
Results saved to: return_risk_results.json
